# Agentic AI — Kafka Pub/Sub with 3 Agents (Publish/Subscribe)




In [48]:
import asyncio
import json
import uuid
from dataclasses import dataclass
from datetime import datetime

from aiokafka import AIOKafkaProducer, AIOKafkaConsumer

from openai import OpenAI
client = OpenAI()

In [49]:
BOOTSTRAP_SERVERS = "localhost:9092"

TOPIC_TASKS = "tasks"
TOPIC_RESULTS = "results"

def now():
    return datetime.utcnow().isoformat() + "Z"

def dumps(obj) -> bytes:
    return json.dumps(obj).encode("utf-8")

def loads(b: bytes):
    return json.loads(b.decode("utf-8"))

In [50]:
@dataclass
class Task:
    task_id: str
    prompt: str

@dataclass
class Result:
    task_id: str
    outcome: str

## Agent implementations

### PlannerAgent (Publisher)
Publishes a few tasks to `tasks`.

### WorkerAgent (Subscriber + Publisher)
Consumes `tasks`, processes them, publishes to `results`.

### MonitorAgent (Subscriber)
Consumes from both topics and prints an audit trail.


In [51]:
class PlannerAgent:
    def __init__(self, producer: AIOKafkaProducer):
        self.producer = producer

    async def run(self):
        prompts = [
            "Classify: 'I was charged twice' => Billing/Technical/Account",
            "Compute: 3 notebooks at $4 and 5 pens at $2, paid $50. Change?",
            "Summarize: What is pub/sub in one sentence?"
        ]
        for p in prompts:
            task_id = str(uuid.uuid4())[:8]
            msg = {"type": "task", "ts": now(), "task_id": task_id, "prompt": p}
            await self.producer.send_and_wait(TOPIC_TASKS, dumps(msg))
            print(f"[PlannerAgent] published task {task_id}")
            await asyncio.sleep(0.5)

In [52]:
class WorkerAgent:
    def __init__(self, consumer: AIOKafkaConsumer, producer: AIOKafkaProducer):
        self.consumer = consumer
        self.producer = producer

    async def process(self, prompt: str) -> str:
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system",
                "content": "You are a helpful assistant agent. Answer tasks briefly."},
                {"role": "user", "content": prompt}
            ]
        )

        return response.choices[0].message.content

    async def run(self):
        async for msg in self.consumer:
            obj = loads(msg.value)
            if obj.get("type") != "task":
                continue
            task_id = obj["task_id"]
            prompt = obj["prompt"]
            outcome = await self.process(prompt)

            res = {"type": "result", "ts": now(), "task_id": task_id, "outcome": outcome}
            await self.producer.send_and_wait(TOPIC_RESULTS, dumps(res))
            print(f"[WorkerAgent] processed task {task_id} -> published result")

In [53]:
class MonitorAgent:
    def __init__(self, consumer_tasks: AIOKafkaConsumer, consumer_results: AIOKafkaConsumer):
        self.consumer_tasks = consumer_tasks
        self.consumer_results = consumer_results

    async def _watch(self, consumer: AIOKafkaConsumer, label: str):
        async for msg in consumer:
            obj = loads(msg.value)
            print(f"[MonitorAgent] saw {label}: {obj}")

    async def run(self):
        await asyncio.gather(
            self._watch(self.consumer_tasks, "TASK"),
            self._watch(self.consumer_results, "RESULT"),
        )

In [54]:
import uuid
import asyncio

async def main(run_seconds: int = 12):
    producer = AIOKafkaProducer(bootstrap_servers=BOOTSTRAP_SERVERS)
    await producer.start()

    # Fresh group ids each run to avoid stale membership/heartbeat noise
    worker_group = f"worker-{uuid.uuid4().hex[:6]}"
    mon_tasks_group = f"monitor-tasks-{uuid.uuid4().hex[:6]}"
    mon_results_group = f"monitor-results-{uuid.uuid4().hex[:6]}"

    worker_consumer = AIOKafkaConsumer(
        TOPIC_TASKS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=worker_group,
        auto_offset_reset="latest",
        enable_auto_commit=False,
    )
    await worker_consumer.start()

    monitor_tasks_consumer = AIOKafkaConsumer(
        TOPIC_TASKS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=mon_tasks_group,
        auto_offset_reset="latest",
        enable_auto_commit=False,
    )
    monitor_results_consumer = AIOKafkaConsumer(
        TOPIC_RESULTS,
        bootstrap_servers=BOOTSTRAP_SERVERS,
        group_id=mon_results_group,
        auto_offset_reset="latest",
        enable_auto_commit=False,
    )
    await monitor_tasks_consumer.start()
    await monitor_results_consumer.start()

    planner = PlannerAgent(producer)
    worker = WorkerAgent(worker_consumer, producer)
    monitor = MonitorAgent(monitor_tasks_consumer, monitor_results_consumer)

    tasks = [
        asyncio.create_task(planner.run()),
        asyncio.create_task(worker.run()),
        asyncio.create_task(monitor.run()),
    ]

    try:
        await asyncio.sleep(run_seconds)
    finally:
        # Cancel tasks cleanly
        for t in tasks:
            t.cancel()
        await asyncio.gather(*tasks, return_exceptions=True)

        # Stop consumers/producers
        await worker_consumer.stop()
        await monitor_tasks_consumer.stop()
        await monitor_results_consumer.stop()
        await producer.stop()

# Run demo:
await main(run_seconds=12)

[PlannerAgent] published task 384ae36d
[MonitorAgent] saw TASK: {'type': 'task', 'ts': '2026-02-02T03:47:20.244229Z', 'task_id': '384ae36d', 'prompt': "Classify: 'I was charged twice' => Billing/Technical/Account"}
[WorkerAgent] processed task 384ae36d -> published result
[MonitorAgent] saw RESULT: {'type': 'result', 'ts': '2026-02-02T03:47:21.828316Z', 'task_id': '384ae36d', 'outcome': 'Billing'}
[PlannerAgent] published task 4dd780ed


Failed fetch messages from 1: [Error 7] RequestTimedOutError
Heartbeat failed: local member_id was not recognized; resetting and re-joining group
Heartbeat session expired - marking coordinator dead
Marking the coordinator dead (node 1)for group monitor-results-5f2e6e.


[MonitorAgent] saw TASK: {'type': 'task', 'ts': '2026-02-02T03:47:21.831070Z', 'task_id': '4dd780ed', 'prompt': 'Compute: 3 notebooks at $4 and 5 pens at $2, paid $50. Change?'}
